# Fish Passage Barrier
## Culvert and Gravity Main Attributes

The purpose of this notebook is to facilitate the development and maintenance of a prioritized list of fish passage barriers.

Note: When running the cells in this notebook, be sure no features are selected in the map. Otherwise, the analyses will only be run on the selected features

In [6]:
import arcpy
arcpy.env.overwriteOutput = True
from datetime import datetime
from IPython.display import display, Markdown

todays_date = str(datetime.now().date())

display(Markdown(f'**Notebook created:** 2024-07-'))
display(Markdown(f'**Last updated:** {todays_date}'))

### Intersect culverts and gravity mains with subbasin polygons

In [24]:
# Culverts
# First, convert assets to points (otherwise assets that cross subbasin boundaries will be duplicated)
arcpy.management.FeatureToPoint(
    in_features="Storm Culverts",
    out_feature_class=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Fish Passage Barriers.gdb\Culvert_points",
    point_location="INSIDE"
)

# Intersect with subbasin
arcpy.analysis.Intersect(
    in_features="'Culvert_points' #;Utilities.UTIL.SD_Basin #",
    out_feature_class=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Fish Passage Barriers.gdb\StormCulverts_Intersect",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)

# Remove unwanted fields
arcpy.management.DeleteField(
    in_table="StormCulverts_Intersect",
    drop_field="Carta_UTIL_swCulvert_FACILITYID;BASIN",
    method="KEEP_FIELDS"
) # Turns out this step actually isn't necessary b/c you can remove unwanted fields when you merge the datasets


<Result 'StormCulverts_Intersect'>

In [18]:
# Gravity Mains
# First, convert assets to points (otherwise assets that cross subbasin boundaries will be duplicated)
arcpy.management.FeatureToPoint(
    in_features="Storm Gravity Mains",
    out_feature_class=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Fish Passage Barriers.gdb\GravityMain_points",
    point_location="INSIDE"
)

arcpy.analysis.Intersect(
    in_features="'GravityMain_points' #;Utilities.UTIL.SD_Basin #",
    out_feature_class=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Fish Passage Barriers.gdb\StormGravityMains_Intersect",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)


<Result 'V:\\UtilitiesDeptGIS\\ArcGIS\\Watershed Planning Team\\Environmental Monitoring\\Fish Passage Barriers\\Fish Passage Barriers\\Fish Passage Barriers.gdb\\StormGravityMains_Intersect'>

In [25]:
# Merge the two datasets
arcpy.management.Merge(
    inputs="StormGravityMains_Intersect;StormCulverts_Intersect",
    output=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Fish Passage Barriers.gdb\StormAssetsByBasin",
    field_mappings='FACILITYID "Asset Number" true true false 20 Text 0 0,First,#,StormGravityMains_Intersect,Carta_UTIL_swGravityMain_FACILITYID,0,20,StormCulverts_Intersect,Carta_UTIL_swCulvert_FACILITYID,0,20;BASIN "BASIN" true true false 20 Text 0 0,First,#,StormGravityMains_Intersect,BASIN,0,20,StormCulverts_Intersect,BASIN,0,20',
    add_source="NO_SOURCE_INFO"
)

<Result 'V:\\UtilitiesDeptGIS\\ArcGIS\\Watershed Planning Team\\Environmental Monitoring\\Fish Passage Barriers\\Fish Passage Barriers\\Fish Passage Barriers.gdb\\StormAssetsByBasin'>

### Get coordinates for each asset

In [26]:
arcpy.management.CalculateGeometryAttributes(
    in_features="StormAssetsByBasin",
    geometry_property="Longitude POINT_X;Latitude POINT_Y",
    length_unit="",
    area_unit="",
    coordinate_system='PROJCS["NAD_1983_2011_StatePlane_Washington_North_FIPS_4601_Ft_US",GEOGCS["GCS_NAD_1983_2011",DATUM["D_NAD_1983_2011",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic"],PARAMETER["False_Easting",1640416.666666667],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-120.8333333333333],PARAMETER["Standard_Parallel_1",47.5],PARAMETER["Standard_Parallel_2",48.73333333333333],PARAMETER["Latitude_Of_Origin",47.0],UNIT["Foot_US",0.3048006096012192]]',
    coordinate_format="DD"
)

<Result 'StormAssetsByBasin'>

### Get stream name and type

In [2]:
# First, snap assets to stream
arcpy.edit.Snap(
    in_features="StormAssetsByBasin",
    snap_environment="Utilities.UTIL.envStreams EDGE '5 Meters'"
)

<Result 'StormAssetsByBasin'>

**Note:** Snapping assets with a 5 m tolerance to the stream means that some assets will be associated with a stream when they don't actually carry the stream. This doesn't matter since we will be using the asset list developed by the consultant that will help us filter those issues out later. However, there is still the possibility that at a confluence a pipe carrying a stream might get snapped to the wrong stream. This will require some QC checks.

In [3]:
# Then, intersect assets with the streams layer via a spatial join to get the stream name and type
arcpy.analysis.SpatialJoin(
    target_features="StormAssetsByBasin",
    join_features="Utilities.UTIL.envStreams",
    out_feature_class=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Fish Passage Barriers.gdb\StormAssetsByBasinAndStream",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping='FACILITYID "Asset Number" true true false 20 Text 0 0,First,#,StormAssetsByBasin,FACILITYID,0,20;BASIN "BASIN" true true false 20 Text 0 0,First,#,StormAssetsByBasin,BASIN,0,20;Shape_Length "Shape_Length" false true true 8 Double 0 0,First,#,StormAssetsByBasin,Shape_Length,-1,-1;Longitude "Longitude" true true false 8 Double 0 0,First,#,StormAssetsByBasin,Longitude,-1,-1;Latitude "Latitude" true true false 8 Double 0 0,First,#,StormAssetsByBasin,Latitude,-1,-1;SDPIPE "Storm Drainage Pipes" true true false 10 Text 0 0,First,#,Utilities.UTIL.envStreams,SDPIPE,0,10;NAME "Name" true true false 25 Text 0 0,First,#,Utilities.UTIL.envStreams,NAME,0,25;SEGMENTID "Segment ID" true true false 20 Text 0 0,First,#,Utilities.UTIL.envStreams,SEGMENTID,0,20;TYPE "Type" true true false 13 Text 0 0,First,#,Utilities.UTIL.envStreams,TYPE,0,13',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name=""
)

<Result 'V:\\UtilitiesDeptGIS\\ArcGIS\\Watershed Planning Team\\Environmental Monitoring\\Fish Passage Barriers\\Fish Passage Barriers\\Fish Passage Barriers.gdb\\StormAssetsByBasinAndStream'>

### Clean it up and export

In [ ]:
# Remove irrelevant columns
arcpy.management.DeleteField(
    in_table="StormAssetsByBasinAndStream",
    drop_field="Join_Count;TARGET_FID",
    method="DELETE_FIELDS"
) # Turns out I could've included this in the export step

# Filter out assets without an asset number and export to csv
arcpy.conversion.ExportTable(
    in_table="StormAssetsByBasinAndStream",
    out_table=r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Output\AssetsByBasinStreamAndType.csv",
    where_clause="FACILITYID IS NOT NULL",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='FACILITYID "Asset Number" true true false 20 Text 0 0,First,#,StormAssetsByBasinAndStream,FACILITYID,0,20;BASIN "BASIN" true true false 20 Text 0 0,First,#,StormAssetsByBasinAndStream,BASIN,0,20;Longitude "Longitude" true true false 8 Double 0 0,First,#,StormAssetsByBasinAndStream,Longitude,-1,-1;Latitude "Latitude" true true false 8 Double 0 0,First,#,StormAssetsByBasinAndStream,Latitude,-1,-1;SDPIPE "Storm Drainage Pipes" true true false 10 Text 0 0,First,#,StormAssetsByBasinAndStream,SDPIPE,0,10;NAME "Name" true true false 25 Text 0 0,First,#,StormAssetsByBasinAndStream,NAME,0,25;SEGMENTID "Segment ID" true true false 20 Text 0 0,First,#,StormAssetsByBasinAndStream,SEGMENTID,0,20;TYPE "Type" true true false 13 Text 0 0,First,#,StormAssetsByBasinAndStream,TYPE,0,13',
    sort_field=None
)



In [21]:
import pandas as pd
# Read it back in if necessary
asset_attributes = pd.read_csv(r"V:\UtilitiesDeptGIS\ArcGIS\Watershed Planning Team\Environmental Monitoring\Fish Passage Barriers\Fish Passage Barriers\Output\AssetsByBasinStreamAndType.csv")
print("There are", asset_attributes.shape[0], "rows in the data frame.")
asset_attributes


There are 27318 rows in the data frame.


,OID_,FACILITYID,BASIN,Longitude,Latitude,SDPIPE,NAME,SEGMENTID,TYPE
0,1,463786,COAL CREEK,-122.128819,47.535547,NaN,NaN,NaN,NaN
1,13,453589,COAL CREEK,-122.115215,47.536071,NaN,NaN,NaN,NaN
2,22,535569,COAL CREEK,-122.115203,47.536140,NaN,NaN,NaN,NaN
3,34,453586,COAL CREEK,-122.115203,47.536233,NaN,NaN,NaN,NaN
4,43,453548,COAL CREEK,-122.115220,47.536380,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
27313,58431,520047,VALLEY CREEK,-122.152641,47.646088,NaN,NaN,NaN,NaN
27314,58433,531350,VALLEY CREEK,-122.156875,47.648677,NaN,NaN,NaN,NaN
27315,58434,531349,VALLEY CREEK,-122.157193,47.648904,NaN,NaN,NaN,NaN
27316,58435,531351,VALLEY CREEK,-122.157238,47.649219,NaN,NaN,NaN,NaN


In [5]:
# List duplicated asset numbers
asset_attributes.FACILITYID[asset_attributes.FACILITYID.duplicated()].unique()

array(['339210', nan, '340333', '334450', '358524'], dtype=object)

### Read in the other datasets

!!! I should break this into two notebooks so that I can re-run the prioritization without running all the GIS things too. 
!!! Pick up here!

In [4]:
import pandas as pd

In [5]:
# The barrier trace results
traceresults = pd.read_csv(r"V:\UtilitiesDeptGIS\Asset Management\Storm\Culvert and Barrier ID Automation Tool\4_PrioritizationFiles\OutputFile\TraceResult.csv")
print(traceresults)

    Structure ID  ...                                Downstream Barriers
0         353780  ...  922157 (Unknown Barrier) -> 921607 (Partial Ba...
1         351811  ...                                                NaN
2         351793  ...                                                NaN
3         354218  ...  922146 (Partial Barrier) -> 922397 (No Barrier...
4         337694  ...  930615 (Partial Barrier) -> 930616 (Partial Ba...
..           ...  ...                                                ...
341       542186  ...  930489 (Full Barrier) -> SB20 (Natural Barrier...
342       513732  ...                                                NaN
343      PARKS-3  ...                                                NaN
344      PARKS-4  ...                                                NaN
345      PARKS-5  ...                                                NaN

[346 rows x 17 columns]


In [10]:
# The prioritization data - culverts
culverts = pd.read_excel(r"V:\UtilitiesDeptGIS\Asset Management\Storm\Culvert and Barrier ID Automation Tool\4_PrioritizationFiles\Prioritization Scoring Input Data.xlsx",
                        sheet_name = 'Environmental Input Data Cul')
print(culverts)

    FACILITYID  ... Accessible channel created Upstream
0       463954  ...                          520.240277
1       539989  ...                            0.000000
2       340855  ...                         2697.572172
3       340862  ...                         2697.572172
4       340870  ...                         2697.572172
..         ...  ...                                 ...
200     540456  ...                            0.000000
201     346346  ...                         1123.896946
202     336163  ...                          523.712203
203     336182  ...                          523.712203
204     342996  ...                         2479.761407

[205 rows x 36 columns]


In [12]:
# The prioritization data - gravity mains
gms = pd.read_excel(r"V:\UtilitiesDeptGIS\Asset Management\Storm\Culvert and Barrier ID Automation Tool\4_PrioritizationFiles\Prioritization Scoring Input Data.xlsx",
                        sheet_name = 'Environmental Input Data GM ')
print(gms)

     FACILITYID  ... Accessible channel created Upstream
0        450501  ...                         1791.370182
1        450498  ...                         1791.370182
2        345880  ...                         2440.415497
3        450504  ...                         1791.370182
4        339177  ...                           63.037718
..          ...  ...                                 ...
136      359871  ...                            0.000000
137      354218  ...                            0.000000
138      356015  ...                            0.000000
139      340182  ...                            0.000000
140      359850  ...                            0.000000

[141 rows x 34 columns]
